# Warp check: does a water-aware transfer of neighbouring photos help? (Colab, T4 is enough)

**No training.** Training-free first test of the idea in project note `water-aware-ibr-residual`: add a residual from neighbouring *training* photos to the splat render, with the underwater physics doing the transfer. For each test view, the two temporal neighbours (Submerged3D) or the two training views that see most of it (SeaThru-NeRF) are warped into it with the model's own depth, keeping only pixels both sources see (occlusion-tested). Six variants are scored on those pixels:

| | variant | what it is |
|---|---|---|
| a | `naive` | neighbour photo copied as is |
| b | `camera` | neighbour's colour correction removed, the test frame's (predicted) applied |
| c | `camera_water` | (b) + water re-applied at the test view's range: `(M - B(1-e^-bB r_s)) e^-bD (r_t - r_s) + B(1-e^-bB r_t)` |
| d | `render` | the current model (= our `neighbors` protocol, 24.76 dB on Submerged3D) |
| e | `render_res_water` | render + water-scaled residual of the neighbour photo against its own render |
| f | `render_res` | render + unscaled residual |

Scores: masked PSNR (8-bit, as evaluate()), masked high-frequency correlation (detail shared with the photo; not charged for the neighbour's noise), and full-image PSNR / SSIM / LPIPS of the composite (variant inside the mask, render outside). Also logged: how much the range changes between neighbour and test view, and how big the water term is in 8-bit levels.

**Pre-registered rule** (written before any real run): *physics*: (c) beats (b) by > 0.1 dB masked PSNR in ≥ 3/4 scenes; *detail*: (c) beats (d) on masked PSNR in ≥ 3/4 scenes (secondary: on HF correlation). Build the residual network only if physics and a detail criterion pass.

Needs only `sfm/`, `images/` and `gs/mvs_lpips/ckpt.pt` per scene on Drive (same as the E2 notebook). About 10 minutes setup, then about a minute per scene.

In [ ]:
# ---- 1. settings ----
DRIVE_DIR = "/content/drive/MyDrive/uwsplat_colab"
SETS = {   # dataset: (outputs root, source choice, test-view colour)
    "Submerged3D": (f"{DRIVE_DIR}/outputs_v2", "temporal", "neighbors"),        # video transects
    "SeaThru-NeRF": (f"{DRIVE_DIR}/outputs_seathru", "overlap", "identity"),    # photo collections (E2: file order != time)
}
SCENES = None                 # None = every scene with gs/<TAG>/ckpt.pt
TAG = "mvs_lpips"
FORCE = False
RES_DRIVE = f"{DRIVE_DIR}/warp_check"
REPO, WORK = "/content/splat_dense-sparse", "/content/warp_scenes"

import os, subprocess, torch
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU (T4 is fine)"
cc = torch.cuda.get_device_capability()
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cc[0]}.{cc[1]}"
print("torch", torch.__version__, "CUDA", torch.version.cuda)

In [ ]:
# ---- 2. Drive ----
from google.colab import drive
drive.mount("/content/drive", timeout_ms=90000)
for dset, (root, _, _) in SETS.items():
    print(f"{dset:13s} {root}:", sorted(os.listdir(root)) if os.path.isdir(root) else "NOT FOUND")
os.makedirs(RES_DRIVE, exist_ok=True)

In [ ]:
%%bash -s "$REPO"
# ---- 3. code (branch dense-v2) and packages ----
set -e
if [ -d "$1/.git" ]; then git -C "$1" fetch -q origin dense-v2 && git -C "$1" reset -q --hard origin/dense-v2
else git clone -q --branch dense-v2 https://github.com/noUpKey/splat_dense-sparse "$1"; fi
git -C "$1" log --oneline -1
test -f "$1/uwsplat/eval/warp_check.py" || { echo "warp_check.py missing: wrong branch?"; exit 1; }
pip -q install "lpips==0.1.4" safetensors ninja "gsplat==1.5.3"

In [ ]:
# ---- 3b. compile gsplat once (5-10 min on a fresh runtime; skip if the C2 notebook already ran in this runtime) ----
import time
t0 = time.time()
from gsplat import rasterization
dev = "cuda"
img, alpha, _ = rasterization(torch.tensor([[0.0, 0.0, 3.0]], device=dev), torch.tensor([[1.0, 0, 0, 0]], device=dev),
                              torch.full((1, 3), 0.1, device=dev), torch.tensor([0.9], device=dev),
                              torch.ones(1, 1, 3, device=dev), torch.eye(4, device=dev)[None],
                              torch.tensor([[[50.0, 0, 32], [0, 50.0, 32], [0, 0, 1]]], device=dev), 64, 64,
                              render_mode="RGB+ED")
print(f"gsplat OK ({time.time() - t0:.0f} s)")

In [ ]:
# ---- 4. CPU check: on a toy textured plane the water transfer reproduces the test photo (+10 dB over camera-only) ----
r = subprocess.run(["python", "-m", "tests.test_warp_check"], cwd=REPO, capture_output=True, text=True)
print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "arn" not in l and "from_numpy" not in l)[-2500:])
assert r.returncode == 0, "toy test failed"

In [ ]:
# ---- 5. scenes on Drive ----
import glob
jobs = []
for dset, (root, srcs, colour) in SETS.items():
    if not os.path.isdir(root):
        continue
    for sd in sorted(glob.glob(f"{root}/*/")):
        sd = sd.rstrip("/")
        scene = os.path.basename(sd)
        if (SCENES and scene not in SCENES) or not os.path.exists(f"{sd}/sfm/scene.json"):
            continue
        has = os.path.exists(f"{sd}/gs/{TAG}/ckpt.pt")
        done = os.path.exists(f"{RES_DRIVE}/{dset}/{scene}/warp_check.json")
        jobs.append(dict(dset=dset, scene=scene, src=sd, has_ckpt=has, done=done, sources=srcs, colour=colour))
        print(f"{dset:13s} {scene:24s} checkpoint: {'yes' if has else 'MISSING'}"
              f"{'   (result on Drive: skip)' if done and not FORCE else ''}")

In [ ]:
# ---- 6. run ----
import json, shutil, sys

def sh(cmd):
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(cmd)}")

failed = []
for j in jobs:
    if not j["has_ckpt"] or (j["done"] and not FORCE):
        continue
    local_root = f"{WORK}/{j['dset']}"
    local = f"{local_root}/{j['scene']}"
    dst = f"{RES_DRIVE}/{j['dset']}/{j['scene']}"
    print(f"\n===== {j['dset']} / {j['scene']}  (sources {j['sources']}, test colour {j['colour']})")
    try:
        for sub in ("sfm", "images"):
            shutil.copytree(f"{j['src']}/{sub}", f"{local}/{sub}", dirs_exist_ok=True)
        os.makedirs(f"{local}/gs/{TAG}", exist_ok=True)
        shutil.copy2(f"{j['src']}/gs/{TAG}/ckpt.pt", f"{local}/gs/{TAG}/ckpt.pt")
        sh([sys.executable, "scripts/warp_check.py", "--out", local_root, "--scenes", j["scene"], "--tag", TAG,
            "--sources", j["sources"], "--target_colour", j["colour"], "--force"])
        os.makedirs(dst, exist_ok=True)
        shutil.copy2(f"{local}/eval/{TAG}/warp_check.json", f"{dst}/warp_check.json")
        shutil.copytree(f"{local}/eval/{TAG}/warp_check", f"{dst}/montages", dirs_exist_ok=True)
        shutil.rmtree(local, ignore_errors=True)
    except Exception as e:
        print("ERROR:", e)
        failed.append(f"{j['dset']}/{j['scene']}")
print("\nfailed:", failed or "none")

## 7. Results and the pre-registered rule
Masked = pixels both neighbours see (occlusion-tested). Montages on Drive (`warp_check/<dataset>/<scene>/montages`): photo | render | camera+water warp | render + water-scaled residual | mask.

In [ ]:
sys.path.insert(0, REPO)
from uwsplat.eval.warp_check import VARIANTS, decide
L = dict(zip(VARIANTS, "abcdef"))
lines = [f"# Warp check ({TAG})", ""]
def out(s=""):
    print(s); lines.append(s)

for dset in SETS:
    files = sorted(glob.glob(f"{RES_DRIVE}/{dset}/*/warp_check.json"))
    if not files:
        continue
    res = {f.split("/")[-2]: json.load(open(f)) for f in files}
    out(f"## {dset}"); out("")
    out("Masked PSNR (HF correlation):"); out("")
    out("| scene | " + " | ".join(f"{L[v]} {v}" for v in VARIANTS) + " | mask | range change | water term |")
    out("|---|" + "---|" * (len(VARIANTS) + 3))
    for sc, r in res.items():
        s = r["summary"]
        if not s:
            out(f"| {sc} | no scored views |"); continue
        out(f"| {sc} | " + " | ".join(f"{s['masked'][v]['psnr']:.2f} ({s['masked'][v]['hf_corr']:.3f})" for v in VARIANTS)
            + f" | {100 * s['mask_fraction']:.0f} % | {100 * s['rel_range_change_median']:.1f} % | "
              f"{s['water_minus_camera_8bit_mean']:.2f} lv |")
    out("")
    out("Full image, composite: PSNR / SSIM / LPIPS"); out("")
    for sc, r in res.items():
        s = r["summary"]
        if s:
            out(f"- {sc}: " + " | ".join(f"{L[v]} {s['full_composite'][v]['psnr']:.2f} / {s['full_composite'][v]['ssim']:.4f} / "
                                         f"{s['full_composite'][v]['lpips']:.4f}" for v in VARIANTS))
    d = decide({sc: r["summary"] for sc, r in res.items()})
    phys = ", ".join(f"{sc} {r['summary']['physics_delta_db']:+.2f}" for sc, r in res.items() if r["summary"])
    det = ", ".join(f"{sc} {r['summary']['detail_delta_db']:+.2f} dB / {r['summary']['detail_delta_hf']:+.3f}"
                    for sc, r in res.items() if r["summary"])
    out("")
    out(f"**Physics** (c) > (b) by > 0.1 dB in ≥ {d['needed']}/{d['n_scenes']}: {d['physics_pass']} ({phys}) -> "
        f"**{'PASS' if d['physics'] else 'FAIL'}**")
    out(f"**Detail** (c) > (d) masked PSNR: {d['detail_psnr_pass']}, HF correlation (secondary): {d['detail_hf_pass']} "
        f"({det}) -> **{'PASS' if d['detail_psnr'] else 'FAIL'}** / {'pass' if d['detail_hf'] else 'fail'}")
    out(f"**Build the residual network ({dset}): {'YES' if d['build_network'] else 'NO'}**")
    out("")

with open(f"{RES_DRIVE}/summary.md", "w") as fh:
    fh.write("\n".join(lines) + "\n")
print(f"saved {RES_DRIVE}/summary.md")